In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import torch
import plotly.express as px


from data_processing import load_and_clean, vocab_sizes
from graph_construction import build_transaction_graph
from feature_engineering import engineer_all_features
from pyg_export import build_pyg_data, NUMERIC_FEATURE_COLUMNS
from neighbor_sampling import SimpleNeighborLoader
from model import LineMVGNN
from train import train_model, run_inference, compute_classification_metrics, print_metrics_report
from aggregation import build_transaction_view, aggregate_accounts

reset_timing_log()
torch.manual_seed(0)
np.random.seed(0)


In [ ]:
TRAIN_CSV = "HI-Small_Trans_SYNTHETIC.csv"   # <-- point at the real HI-Small_Trans.csv when available
TEST_CSV = "LI-Small_Trans_SYNTHETIC.csv"    # <-- point at the real LI-Small_Trans.csv when available


pd.read_csv(TRAIN_CSV, nrows=5)


HI-Small_Trans_SYNTHETIC.csv not found -- generating a synthetic stand-in with the IBM-AML schema.


LI-Small_Trans_SYNTHETIC.csv not found -- generating a synthetic stand-in with the IBM-AML schema.


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:03:37.660963980,25,ACC100794,36,ACC100955,2268.04,Yen,2268.04,Yen,ACH,0
1,2022-09-01 00:06:22.640417406,44,ACC100390,25,ACC100953,99.17,UK Pound,99.17,UK Pound,ACH,0
2,2022-09-01 00:10:29.276885352,30,ACC100343,4,ACC100126,2660.08,US Dollar,2660.08,US Dollar,Wire,0
3,2022-09-01 00:12:54.889506792,47,ACC100483,0,ACC101278,87.59,UK Pound,87.59,UK Pound,Reinvestment,0
4,2022-09-01 00:16:32.754439974,30,ACC100896,6,ACC100257,1175.10,Yen,1175.10,Yen,Reinvestment,0


In [3]:
df_train, vocabs = load_and_clean(TRAIN_CSV)
print(f"Train: {len(df_train):,} transactions, laundering rate {df_train['label'].mean():.3%}")
print(f"Memory (deep): {df_train.memory_usage(deep=True).sum() / 1e9:.3f} GB")
vocab_sizes(vocabs)


[TIMER] 01. Data Loading & Normalization                 0.163s
Train: 40,000 transactions, laundering rate 4.500%
Memory (deep): 0.002 GB


{'n_accounts': 1501, 'n_banks': 61, 'n_payment_formats': 7, 'n_currencies': 7}

In [4]:
g_train, src_train, dst_train, preds_train, succs_train = build_transaction_graph(df_train)
print(f"Train graph: {g_train.numberOfNodes():,} nodes, {g_train.numberOfEdges():,} edges "
      f"(avg out-degree {g_train.numberOfEdges()/g_train.numberOfNodes():.2f})")


[TIMER] 02. Transaction Graph Construction (NetworKit)    0.085s
Train graph: 40,000 nodes, 164,063 edges (avg out-degree 4.10)


In [5]:
df_train = engineer_all_features(df_train, preds_train, succs_train, src_train, dst_train)
print(f"Engineered feature count: {len(NUMERIC_FEATURE_COLUMNS)} numeric + 7 categorical embedding inputs")
df_train.filter(regex="^(sender_|receiver_|pair_|fan_|relay_|short_cycle)").head()


[TIMER] 03a. Feature Engineering - Temporal Features     0.026s
[TIMER] 03b. Feature Engineering - Pair History Features    0.058s


[TIMER] 03c. Feature Engineering - Transaction Flow Features    1.036s


[TIMER] 03d. Feature Engineering - Account Context Features    1.883s
Engineered feature count: 39 numeric + 7 categorical embedding inputs


,sender_idx,receiver_idx,pair_prior_txn_count,pair_total_amount_prior,pair_mean_amount_prior,pair_std_amount_prior,pair_max_amount_prior,pair_repeated_exact_amount_count,fan_in,fan_out,...,sender_velocity_1d,sender_velocity_10d,receiver_hist_outflow_total,receiver_hist_inflow_total,receiver_unique_counterparties,receiver_entropy,receiver_concentration,receiver_degree_balance,receiver_velocity_1d,receiver_velocity_10d
0,794,955,0,0.0,0.0,0.0,0.0,0,0,4,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
1,390,953,0,0.0,0.0,0.0,0.0,0,0,4,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
2,343,126,0,0.0,0.0,0.0,0.0,0,0,1,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
3,483,1278,0,0.0,0.0,0.0,0.0,0,0,6,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0
4,896,257,0,0.0,0.0,0.0,0.0,0,0,1,...,0,0,0.0,0.0,0,0.0,0.0,0.0,0,0


In [6]:
data_train, scaler = build_pyg_data(df_train, src_train, dst_train, fit_scaler=True)
print(data_train)


[TIMER] 04. Graph Export to PyTorch Geometric            0.062s
Data(x=[40000, 39], edge_index=[2, 164063], y=[40000], num_nodes=40000, sender_idx=[40000], receiver_idx=[40000], from_bank_idx=[40000], to_bank_idx=[40000], payment_format_idx=[40000], payment_currency_idx=[40000], receiving_currency_idx=[40000])


In [7]:
vs = vocab_sizes(vocabs)
model = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    n_accounts=vs["n_accounts"],
    n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"],
    n_currencies=vs["n_currencies"],
    emb_dim=16, hidden_dim=64, num_layers=2, dropout=0.3,
)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


LineMVGNN(
  (account_emb): Embedding(1501, 16)
  (bank_emb): Embedding(61, 8)
  (payfmt_emb): Embedding(7, 8)
  (currency_emb): Embedding(7, 8)
  (input_proj): Linear(in_features=111, out_features=64, bias=True)
  (gat_layers): ModuleList(
    (0-1): 2 x GATConv(64, 16, heads=4)
  )
  (gcn_layers): ModuleList(
    (0-1): 2 x GCNConv(64, 64)
  )
  (attr_mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (fusion): Linear(in_features=192, out_features=64, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)
Trainable parameters: 71,465


In [ ]:
model, history = train_model(
    model, 
    data_train, 
    preds_train,
    num_epochs=8, 
    batch_size=512, 
    num_neighbors=(15, 10),
    lr=1e-3, 
    weight_decay=1e-4, 
    val_frac=0.15,
    neg_per_pos=70,
)

fig = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y=["train_loss", "val_loss"],
    title="Training / validation loss curves",
    labels={"value": "BCE loss", "variable": "split"},
)
fig.show()

fig2 = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y="val_pr_auc", title="Validation PR-AUC by epoch",
)
fig2.show()


Train pos_weight (class-imbalance correction): 20.98 (1547 positive / 32453 negative)


Epoch  1/8  train_loss=0.9514  val_loss=0.3879  val_PR-AUC=0.7002  lr=1.00e-03


Epoch  2/8  train_loss=0.2683  val_loss=0.2725  val_PR-AUC=0.7531  lr=1.00e-03


Epoch  3/8  train_loss=0.2155  val_loss=0.2997  val_PR-AUC=0.7317  lr=1.00e-03


Epoch  4/8  train_loss=0.1979  val_loss=0.2906  val_PR-AUC=0.7296  lr=1.00e-03


Epoch  5/8  train_loss=0.1823  val_loss=0.2750  val_PR-AUC=0.7547  lr=1.00e-03


Epoch  6/8  train_loss=0.1590  val_loss=0.2941  val_PR-AUC=0.7504  lr=1.00e-03


Epoch  7/8  train_loss=0.1643  val_loss=0.3697  val_PR-AUC=0.7583  lr=1.00e-03


Epoch  8/8  train_loss=0.1452  val_loss=0.3432  val_PR-AUC=0.7597  lr=1.00e-03
[TIMER] 05. Model Training (total)                     111.433s


In [9]:
import pickle

torch.save(model.state_dict(), "line_mvgnn_weights.pth")
with open("preprocessing_state.pkl", "wb") as f:
    pickle.dump({"scaler": scaler, "vocabs": vocabs}, f)
print("Saved line_mvgnn_weights.pth and preprocessing_state.pkl")

del g_train, src_train, dst_train, preds_train, succs_train, df_train, data_train
gc.collect()
print("Freed train's raw graph/feature objects. "
      "Only `model`, `scaler`, and `vocabs` carry forward from here.")


Saved line_mvgnn_weights.pth and preprocessing_state.pkl


Freed train's raw graph/feature objects. Only `model`, `scaler`, and `vocabs` carry forward from here.


In [10]:
df_test, _ = load_and_clean(TEST_CSV, vocabs=vocabs)
print(f"Test: {len(df_test):,} transactions, laundering rate {df_test['label'].mean():.3%}")
print(f"Memory (deep): {df_test.memory_usage(deep=True).sum() / 1e9:.3f} GB")


[TIMER] 01. Data Loading & Normalization                 0.091s
Test: 15,000 transactions, laundering rate 4.500%
Memory (deep): 0.001 GB


In [11]:
g_test, src_test, dst_test, preds_test, succs_test = build_transaction_graph(df_test)
print(f"Test graph: {g_test.numberOfNodes():,} nodes, {g_test.numberOfEdges():,} edges")

df_test = engineer_all_features(df_test, preds_test, succs_test, src_test, dst_test)
data_test, _ = build_pyg_data(df_test, src_test, dst_test, scaler=scaler, fit_scaler=False)
print(data_test)


[TIMER] 02. Transaction Graph Construction (NetworKit)    0.050s
Test graph: 15,000 nodes, 29,036 edges
[TIMER] 03a. Feature Engineering - Temporal Features     0.017s


[TIMER] 03b. Feature Engineering - Pair History Features    0.036s


[TIMER] 03c. Feature Engineering - Transaction Flow Features    0.240s


[TIMER] 03d. Feature Engineering - Account Context Features    0.425s
[TIMER] 04. Graph Export to PyTorch Geometric            0.009s
Data(x=[15000, 39], edge_index=[2, 29036], y=[15000], num_nodes=15000, sender_idx=[15000], receiver_idx=[15000], from_bank_idx=[15000], to_bank_idx=[15000], payment_format_idx=[15000], payment_currency_idx=[15000], receiving_currency_idx=[15000])


In [12]:
test_probs = run_inference(model, data_test, preds_test, batch_size=1024, num_neighbors=(15, 10))
test_y = data_test.y.numpy()

metrics = compute_classification_metrics(test_y, test_probs, threshold=0.5)
print_metrics_report(metrics)

cm = metrics["confusion_matrix"]
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Legitimate (0)", "Laundering (1)"], y=["Legitimate (0)", "Laundering (1)"],
    title="Confusion matrix - test set",
)
fig.show()


[TIMER] 06. Model Inference                              0.398s

                      TEST SET METRICS                      
accuracy    : 0.9668
precision   : 0.5791
recall      : 0.9600
f1          : 0.7224
pr_auc      : 0.8652
Confusion matrix [rows=true, cols=pred] (0=legit, 1=laundering):
[[13854   471]
 [   27   648]]



In [13]:
reloaded = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    n_accounts=vs["n_accounts"], n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"], n_currencies=vs["n_currencies"],
)
reloaded.load_state_dict(torch.load("line_mvgnn_weights.pth"))
reloaded.eval()
probs_reloaded = run_inference(reloaded, data_test, preds_test, batch_size=1024, num_neighbors=(15, 10))
print("Max abs difference vs. original model's probabilities:", np.abs(test_probs - probs_reloaded).max())


[TIMER] 06. Model Inference                              0.399s
Max abs difference vs. original model's probabilities: 0.0


In [14]:
RISK_THRESHOLD = 0.5

transaction_view = build_transaction_view(df_test, test_probs, threshold=RISK_THRESHOLD)
account_view = aggregate_accounts(df_test, test_probs, threshold=RISK_THRESHOLD)

transaction_view.to_csv("transaction_view.csv", index=False)
account_view.to_csv("account_view.csv", index=False)

print(f"{transaction_view['Flagged'].sum():,} / {len(transaction_view):,} transactions flagged "
      f"at threshold {RISK_THRESHOLD}")
print(f"{account_view['Account Alert'].sum():,} / {len(account_view):,} accounts alerted")
transaction_view.head(10)


[TIMER] 07. Account Risk Aggregation                     0.021s
1,119 / 15,000 transactions flagged at threshold 0.5
908 / 1,200 accounts alerted


,Transaction ID,Risk Score,Sender,Receiver,Amount,Timestamp,Flagged
0,13148,1.000000,ACC100575,ACC101125,6062.589844,2022-10-23 14:12:32.323269531,True
1,2109,1.000000,ACC101001,ACC100352,6771.750000,2022-09-09 01:15:17.699444013,True
2,2669,0.999999,ACC100160,ACC100870,4552.169922,2022-09-11 09:10:48.119794789,True
3,11551,0.999999,ACC100623,ACC100884,7799.419922,2022-10-17 03:32:15.173688692,True
4,7581,0.999997,ACC100359,ACC101011,6796.930176,2022-10-01 04:30:15.926275463,True
5,13685,0.999996,ACC100246,ACC101003,4782.000000,2022-10-25 14:48:20.671920491,True
6,2294,0.999994,ACC100583,ACC100951,4326.209961,2022-09-09 18:52:44.077729642,True
7,1508,0.999992,ACC100076,ACC100782,4294.290039,2022-09-06 18:24:27.862495022,True
8,12455,0.999989,ACC100686,ACC101024,3990.949951,2022-10-20 22:01:34.696478070,True
9,10665,0.999987,ACC100967,ACC101104,7529.310059,2022-10-13 14:38:18.817485176,True


In [15]:
account_view.head(10)

,Account ID,Associated Risk Score,Mean Transaction Risk,Number of Flagged Transactions,Total Transactions,Counterparty Summary,Account Alert
0,ACC100575,1.000000,0.124935,4,30,28,True
1,ACC101125,1.000000,0.166390,3,18,17,True
2,ACC101001,1.000000,0.245536,10,41,37,True
3,ACC100352,1.000000,0.090622,2,26,24,True
4,ACC100160,0.999999,0.169525,5,28,27,True
5,ACC100870,0.999999,0.122156,2,16,15,True
6,ACC100623,0.999999,0.117908,5,38,36,True
7,ACC100884,0.999999,0.173610,4,23,21,True
8,ACC101011,0.999997,0.155293,5,30,27,True
9,ACC100359,0.999997,0.206065,6,29,27,True
